In [1]:
# Step 1: Ingest a document and create embeddings
from src.chunk_embed_store import ingest_document
import os

file_path = os.path.join("data", "Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile.pdf")
config_path = "config.yaml"

print("Step 1: Ingesting document and creating embeddings...")
num_chunks = ingest_document(file_path, config_path)
print(f"✓ Successfully created {num_chunks} chunks with embeddings")

Step 1: Ingesting document and creating embeddings...
Creating ChromaDB store at: ./chroma_db (collection: documents_nomic_embed_text)
ChromaDB created at: C:\Users\user\OneDrive\Desktop\Bot Consulting Assignment\rag_system_v0.1\chroma_db
Successfully ingested data\Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile.pdf
✓ Successfully created 546 chunks with embeddings


In [2]:
# Verify number of chunks created
print(f"Total chunks created: {num_chunks}")

Total chunks created: 546


In [5]:
# Step 2: Use retriever agent to retrieve relevant chunks
from src.retriever_agent import RetrieverAgent

print("\nStep 2: Initializing retriever agent...")
agent = RetrieverAgent(config_path="config.yaml")
print("✓ Retriever agent initialized")

# Retrieve chunks for a query
query = "How can you securely let an Azure Web App (App Service) access Azure Blob Storage without hardcoding the Storage Account key in the application configuration?"
print(f"\nQuery: {query}")
print("Retrieving relevant chunks...")

result = agent.retrieve(query)

print(f"\n✓ Retrieved {len(result.chunks)} relevant chunks:")
print("-" * 80)

for i, chunk in enumerate(result.chunks, 1):
    print(f"\n{i}. Chunk ID: {chunk.chunk_id}")
    print(f"   Rerank Score: {chunk.rerank_score:.4f}")
    print(f"   Text Preview: {chunk.text[:200]}...")
    if chunk.metadata:
        print(f"   Metadata: {chunk.metadata}")


Step 2: Initializing retriever agent...
✓ Retriever agent initialized

Query: How can you securely let an Azure Web App (App Service) access Azure Blob Storage without hardcoding the Storage Account key in the application configuration?
Retrieving relevant chunks...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 893.98it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✓ Retrieved 8 relevant chunks:
--------------------------------------------------------------------------------

1. Chunk ID: Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile_page255_chunk0
   Rerank Score: 0.9736
   Text Preview: 224 of 540 C H A P T E R 4  |  Azure Storage 
 
enables you to secure your data without having 
to add any code to any of your applications. 
Note that it only works for blob storage; tables, 
queues,...
   Metadata: {'file_name': 'Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile.pdf', 'page_num': 255, 'source_unit': 'page', 'file_path': 'data\\Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile.pdf', 'file_type': '.pdf', 'chunk_local_index': 0}

2. Chunk ID: Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile_page250_chunk0
   Rerank Score: 0.9641
   Text Preview: 219 of 540 C H A P T E R 4  |  Azure Storage 
 
putting the keys in the web.config file where a 
hacker could get to them.  
Securing access to yo

### Step 3: Full RAG flow — query LLM and get grounded response

Use **GroundedRAGPipeline** to run retrieval + response generation in one call. The pipeline uses the configured LLM (e.g. **llama3.2** via Ollama) to:
1. Optionally ground the user question using conversation history.
2. Generate a final answer from the retrieved chunks only (with chunk citations).

In [4]:
# Step 3: Full RAG flow — GroundedRAGPipeline (retrieve + LLM response)
from src.response_generator import GroundedRAGPipeline

print("Step 3: Initializing grounded RAG pipeline (retriever + response generator)...")
pipeline = GroundedRAGPipeline(config_path="config.yaml")
# Session ID for conversation history (e.g. user or session identifier)
session_id = "test-session-1"
question = "How can you securely let an Azure Web App (App Service) access Azure Blob Storage without hardcoding the Storage Account key in the application configuration?"

print(f"\nQuery: {question}")
print("Running: retrieve → ground question → generate grounded answer (LLM: llama3.2)...\n")

response = pipeline.ask(session_id=session_id, question=question)

print("=" * 80)
print("GROUNDED RESPONSE")
print("=" * 80)
print(f"Session ID:        {response.session_id}")
print(f"User question:     {response.user_question}")
print(f"Enhanced query:     {response.enhanced_query}")
print(f"Context chunk IDs:  {response.context_chunk_ids}")
print("-" * 80)
print("Final answer:")
print(response.response)
print("=" * 80)

Step 3: Initializing grounded RAG pipeline (retriever + response generator)...

Query: How can you securely let an Azure Web App (App Service) access Azure Blob Storage without hardcoding the Storage Account key in the application configuration?
Running: retrieve → ground question → generate grounded answer (LLM: llama3.2)...



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 887.46it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GROUNDED RESPONSE
Session ID:        test-session-1
User question:     How can you securely let an Azure Web App (App Service) access Azure Blob Storage without hardcoding the Storage Account key in the application configuration?
Enhanced query:     Let's rewrite the latest user question into a standalone grounded retrieval query.

Rewritten query:
("Azure Web App" AND "access" AND "Azure Blob Storage") AND NOT ("hardcoded" AND "Storage Account key")
Context chunk IDs:  ['Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile_page255_chunk0', 'Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile_page250_chunk0', 'Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile_page292_chunk0', 'Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile_page247_chunk0', 'Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile_page287_chunk0', 'Microsoft Azure Essentials Fundamentals of Azure 2nd ed mobile_page249_chunk0', 'Microsoft Azure Essentials Fundamenta